# QDM Smoke Test: Quantization × SAE Features**Goal:** Verify the full pipeline works end-to-end on a tiny scale before committing to vast.ai.**What this notebook does:**1. Loads Pythia-70m-deduped (small Pythia model, 6 layers)2. Loads a pretrained SAE from sae_lens for one layer of that model3. Caches activations from the FP16 model on 50k tokens of WikiText-24. Simulates INT8 quantization by round-to-nearest on the weights5. Caches activations from the quantized model on the same tokens6. Runs the SAE on both, gets per-feature activations7. Computes Pearson correlation per feature between FP16 and INT88. Plots the histogram (the key smoke-test output)9. Inspects the most disrupted features**Expected output:** A histogram where most features have correlation > 0.9 (survived) with a tail toward lower correlations (disrupted). If you see this shape, the pipeline works.**Why simulated quantization instead of bitsandbytes:** Easier to debug, plays nicely with TransformerLens which is what the SAE expects. We'll graduate to real bitsandbytes/GPTQ in Phase 2 on vast.ai.**Runtime:** ~5-15 minutes on a T4 once everything installs.

## 1. Install dependenciesRun this cell once. Restart the runtime if Colab asks you to.

In [ ]:
!pip install -q transformer_lens sae-lens datasets matplotlibprint("Done.")

## 2. GPU check and importsVerify you have a GPU. T4 is fine. If you see "no GPU", change Runtime → Change runtime type → GPU.

In [ ]:
import torchprint("CUDA available:", torch.cuda.is_available())if torch.cuda.is_available():    print("Device:", torch.cuda.get_device_name(0))    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")else:    raise RuntimeError("No GPU. Set Runtime → Change runtime type → GPU.")import numpy as npimport matplotlib.pyplot as pltfrom datasets import load_datasetfrom transformer_lens import HookedTransformerfrom sae_lens import SAEfrom tqdm.auto import tqdmdevice = "cuda"torch.set_grad_enabled(False)  # we only do inferenceprint("Imports OK.")

## 3. Load Pythia-70m-deduped via TransformerLensTransformerLens loads the model with its standard pre-processing (layernorm folding, etc.). The pretrained SAE we'll load was trained on these exact activations, so we must use TL here.First run downloads weights (~280MB). Takes 1-2 min.

In [ ]:
MODEL_NAME = "pythia-70m-deduped"model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)model.eval()# Pythia-70m: 6 layers (0-5), d_model=512, vocab=50304print(f"Model loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")print(f"Tokenizer vocab: {model.cfg.d_vocab}")

## 4. Get text dataWe use the test split of WikiText-2. Take enough text to give us 50k tokens for the smoke test.

In [ ]:
SMOKE_TOKENS = 50_000SEQ_LEN = 512  # length of each sequence we'll processds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")text_chunks = [x for x in ds["text"] if len(x.strip()) > 100]full_text = "\n\n".join(text_chunks)print(f"Total characters: {len(full_text):,}")tokens = model.to_tokens(full_text, prepend_bos=False)[0]print(f"Total tokens: {tokens.shape[0]:,}")# Trim to SMOKE_TOKENS and reshape into (n_seqs, SEQ_LEN)n_seqs = SMOKE_TOKENS // SEQ_LENtokens_smoke = tokens[:n_seqs * SEQ_LEN].reshape(n_seqs, SEQ_LEN).to(device)print(f"Smoke test tensor shape: {tokens_smoke.shape}")

## 5. Load pretrained SAEWe use Joseph Bloom's pretrained SAEs for Pythia-70m at the residual stream. Layer 4 is a good mid-late choice (model has layers 0-5).

In [ ]:
LAYER = 4HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"sae, cfg_dict, sparsity = SAE.from_pretrained(    release="pythia-70m-deduped-res-sm",    sae_id=HOOK_NAME,    device=device,)sae.eval()print(f"SAE loaded for {HOOK_NAME}")print(f"  d_in:  {sae.cfg.d_in}")print(f"  d_sae: {sae.cfg.d_sae}  (expansion factor {sae.cfg.d_sae // sae.cfg.d_in}x)")

## 6. Sanity check: does the SAE produce sensible features?Run the model on a few sentences, encode with the SAE, find the highest-activating feature, and show what tokens it fires on. If the SAE is loaded correctly, you should see thematically related tokens. If you see random garbage, something is wrong.

In [ ]:
sanity_text = (    "The quick brown fox jumps over the lazy dog. "    "Python is a popular programming language used in machine learning. "    "The president signed the bill into law yesterday.")sanity_tokens = model.to_tokens(sanity_text, prepend_bos=True).to(device)_, sanity_cache = model.run_with_cache(sanity_tokens, names_filter=[HOOK_NAME])sanity_acts = sanity_cache[HOOK_NAME][0]  # (seq, d_model)sanity_features = sae.encode(sanity_acts)  # (seq, d_sae)# Find the feature that fires most overall in this snippetmax_per_feature = sanity_features.max(dim=0).valuestop_feat = max_per_feature.argmax().item()print(f"Top firing feature for sanity text: feature {top_feat}")# Show its activations on each tokenstr_tokens = model.to_str_tokens(sanity_tokens[0])feat_acts = sanity_features[:, top_feat].cpu().numpy()print(f"\nActivations of feature {top_feat} across tokens:")for tok, act in zip(str_tokens, feat_acts):    bar = "█" * int(act * 5) if act > 0 else ""    print(f"  {repr(tok):<15} {act:6.2f}  {bar}")print("\nIf the highest activations are on related tokens, the SAE is working.")

## 7. Cache FP16 activations on 50k tokensThis is the reference set we'll compare quantized activations against.

In [ ]:
def cache_residual_stream(model, tokens_2d, hook_name, batch_size=16):    """Run model on tokens, return (n_total_tokens, d_model) activations at hook_name."""    storage = []    n_seqs = tokens_2d.shape[0]    for i in tqdm(range(0, n_seqs, batch_size), desc=f"Caching {hook_name}"):        batch = tokens_2d[i:i+batch_size]        _, cache = model.run_with_cache(batch, names_filter=[hook_name])        storage.append(cache[hook_name].cpu())  # (batch, seq, d_model)    acts = torch.cat(storage, dim=0)  # (n_seqs, seq_len, d_model)    return acts.reshape(-1, acts.shape[-1])  # (n_total_tokens, d_model)acts_fp16 = cache_residual_stream(model, tokens_smoke, HOOK_NAME)print(f"\nFP16 activations shape: {acts_fp16.shape}")print(f"Mean: {acts_fp16.mean():.3f}, Std: {acts_fp16.std():.3f}")

## 8. Simulate INT8 quantizationWe do round-to-nearest symmetric quantization on each weight matrix's parameters. This is the simplest possible quantization and serves as a smoke test. Real GPTQ/AWQ/bitsandbytes would compensate for outliers differently — but for "does the pipeline work," RTN is enough.We save the original weights first, modify them in place, then restore at the end.

In [ ]:
# Save original weights so we can restore afteroriginal_state = {k: v.clone() for k, v in model.state_dict().items()}# Apply RTN INT8 quantization to all weight matrices (not biases, not embeddings)# We quantize per-tensor symmetric: scale = max(|w|) / 127, q = round(w/scale).clamp(-128, 127)quantized_params = 0for name, param in model.named_parameters():    # Quantize only the linear layer weights (W_Q, W_K, W_V, W_O, W_in, W_out)    # Skip embeddings, layernorms, biases, unembed    if any(s in name for s in ["W_Q", "W_K", "W_V", "W_O", "W_in", "W_out"]):        w = param.data        scale = w.abs().max() / 127.0        q = torch.round(w / scale).clamp(-128, 127)        param.data = (q * scale).to(w.dtype)        quantized_params += w.numel()print(f"Quantized {quantized_params:,} parameters across {sum(1 for n,_ in model.named_parameters() if any(s in n for s in ['W_Q','W_K','W_V','W_O','W_in','W_out']))} tensors.")

## 9. Cache "INT8" activations on the same tokens

In [ ]:
acts_int8 = cache_residual_stream(model, tokens_smoke, HOOK_NAME)print(f"\nINT8 activations shape: {acts_int8.shape}")print(f"Mean: {acts_int8.mean():.3f}, Std: {acts_int8.std():.3f}")# Activation-level drift (sanity check)mse = (acts_fp16 - acts_int8).pow(2).mean().item()cos = torch.nn.functional.cosine_similarity(    acts_fp16.flatten().unsqueeze(0),    acts_int8.flatten().unsqueeze(0)).item()print(f"\nActivation MSE: {mse:.4f}")print(f"Activation cosine similarity: {cos:.4f}")print("(Cosine close to 1.0 = mild perturbation, as expected for INT8 RTN)")# Restore original weights so the model isn't permanently quantizedmodel.load_state_dict(original_state)print("Original FP16 weights restored.")

## 10. Apply SAE to both sets of activationsThe SAE encoder gives us sparse features. We get one feature vector per token in each condition.

In [ ]:
def encode_in_batches(sae, acts, batch=8192):    """Encode activations through the SAE in batches to avoid OOM."""    out = []    for i in tqdm(range(0, acts.shape[0], batch), desc="SAE encoding"):        chunk = acts[i:i+batch].to(device).float()        out.append(sae.encode(chunk).cpu())    return torch.cat(out, dim=0)features_fp16 = encode_in_batches(sae, acts_fp16)features_int8 = encode_in_batches(sae, acts_int8)print(f"\nFeatures FP16: {features_fp16.shape}")print(f"Features INT8: {features_int8.shape}")

## 11. Compute per-feature Pearson correlationsFor each feature, correlate its per-token activations between FP16 and INT8. High correlation = "this feature survived." Low correlation = "this feature got disrupted."We only look at features that actually fire in FP16 (firing rate > 0.1%). Features that never fire don't tell us anything about damage.

In [ ]:
def per_feature_pearson(a, b, eps=1e-8):    """Per-feature Pearson correlation. a, b both (N, F)."""    a_centered = a - a.mean(dim=0, keepdim=True)    b_centered = b - b.mean(dim=0, keepdim=True)    num = (a_centered * b_centered).sum(dim=0)    denom = torch.sqrt((a_centered**2).sum(dim=0) * (b_centered**2).sum(dim=0)) + eps    return num / denomcorrelations = per_feature_pearson(features_fp16, features_int8)# Mask to active featuresfp16_firing_rate = (features_fp16 > 0).float().mean(dim=0)active_mask = fp16_firing_rate > 0.001active_corrs = correlations[active_mask]print(f"Total SAE features: {correlations.shape[0]:,}")print(f"Active in FP16 (firing > 0.1%): {active_mask.sum().item():,}")print(f"\nCorrelation statistics for active features:")print(f"  Mean:    {active_corrs.mean():.3f}")print(f"  Median:  {active_corrs.median():.3f}")print(f"  Min:     {active_corrs.min():.3f}")print(f"  Max:     {active_corrs.max():.3f}")print(f"\nSurvival breakdown:")print(f"  > 0.9 (survived):   {(active_corrs > 0.9).float().mean():.1%}")print(f"  0.5–0.9 (degraded): {((active_corrs > 0.5) & (active_corrs <= 0.9)).float().mean():.1%}")print(f"  < 0.5 (damaged):    {(active_corrs < 0.5).float().mean():.1%}")

## 12. Plot the histogramThis is the headline output. What you want to see:- **Most features near 1.0:** quantization is mild, most features survive- **A tail toward lower correlations:** some features are disrupted, the interesting cases- **A few features near 0 or negative:** these are the "damaged" features worth inspectingIf the entire histogram is at 1.0, quantization didn't actually happen. If it's uniform across [0, 1], something is wrong with the comparison. If it's bimodal with one peak at 1.0 and one at 0.0, that's actually interesting and would mean strong feature-specific effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].hist(active_corrs.numpy(), bins=60, edgecolor='black', alpha=0.8)axes[0].set_xlabel("Per-feature Pearson correlation (FP16 vs INT8)")axes[0].set_ylabel("Number of features")axes[0].set_title(f"Feature correlation distribution\nPythia-70m layer {LAYER}, {SMOKE_TOKENS:,} tokens, RTN INT8")axes[0].axvline(0.9, color='green', linestyle='--', alpha=0.7, label='Survival (0.9)')axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Damage (0.5)')axes[0].legend()axes[0].grid(True, alpha=0.3)# Log-scale version to see the tailaxes[1].hist(active_corrs.numpy(), bins=60, edgecolor='black', alpha=0.8)axes[1].set_xlabel("Per-feature Pearson correlation (FP16 vs INT8)")axes[1].set_ylabel("Number of features (log)")axes[1].set_title("Same distribution, log y-axis (highlights tail)")axes[1].set_yscale('log')axes[1].axvline(0.9, color='green', linestyle='--', alpha=0.7)axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.7)axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 13. Inspect the most disrupted featuresFind the features that fired strongly in FP16 but had low correlation with their INT8 counterparts. Show their top activating contexts. If the contexts look semantically coherent (e.g., all about a specific topic), this is a real feature that quantization disrupted — not random noise.If the most-disrupted features look like nonsense, that's actually a clue: weak/poorly-defined features are the first to break.

In [ ]:
# Active feature indices, ordered by correlation (ascending)active_idx = torch.where(active_mask)[0]active_corrs_for_sorting = correlations[active_idx]ordered = active_idx[active_corrs_for_sorting.argsort()]print("=== 10 most disrupted active features ===\n")flat_tokens = tokens_smoke.flatten().cpu()for rank, feat_idx in enumerate(ordered[:10]):    feat_idx = feat_idx.item()    corr = correlations[feat_idx].item()    firing = fp16_firing_rate[feat_idx].item()    print(f"#{rank+1}  feature {feat_idx}   corr={corr:+.3f}   FP16 firing rate={firing:.4f}")        # Top-3 activating tokens for this feature in FP16    feat_acts_fp16 = features_fp16[:, feat_idx]    top_positions = feat_acts_fp16.argsort(descending=True)[:3]    for pos in top_positions:        pos = pos.item()        start = max(0, pos - 12)        end = min(len(flat_tokens), pos + 3)        context = model.tokenizer.decode(flat_tokens[start:end])        # Mark the activating token        marker_pos = pos - start        toks_around = model.to_str_tokens(flat_tokens[start:end])        marked = "".join(            f"[{t}]" if i == marker_pos else t             for i, t in enumerate(toks_around)        )        print(f"    act={feat_acts_fp16[pos]:.2f}: {marked}")    print()

## 14. What this output means and what to do next**If your histogram has the expected shape** (most features near 1.0, tail toward lower correlations, a few near 0):- The pipeline works. You can scale to Phase 2.- Note the fraction of features with correlation < 0.5 — this is your "damage rate" and gives you a sense of effect size.- The most-disrupted features should look like real features (coherent activating contexts), not noise.**If your histogram is all 1.0:**- Quantization didn't actually happen. Check that the weight modification cell ran before caching.**If your histogram is uniform or all near 0:**- The FP16 and INT8 activations aren't comparable. Most likely cause: tokens were re-shuffled between the two passes. Check that `tokens_smoke` is the same tensor in both `cache_residual_stream` calls.**If the most-disrupted features have incoherent contexts:**- Probably fine — weak features should break first. Look at features ranked #20-30 by disruption; those should be more coherent.**Next steps for Phase 2 (vast.ai):**- Scale to Pythia-410M (and its SAEs from sae_lens or train fresh)- Replace simulated RTN with real bitsandbytes INT8 and GPTQ/AWQ INT4- Cache 1M tokens per condition- Add multiple bit-widths (FP16, INT8, INT6, INT4, INT3)- Add the retrained-SAE probe for measuring feature merging- Run 5 seeds- Add the pruning baseline for differentiation from Borobia et al.**One sanity check before scaling up:** also compute the perplexity change. Open a new cell, run model on `tokens_smoke` in both states, compute average loss. If FP16 perplexity ≈ INT8 perplexity (within 5%), you've confirmed the model still works after quantization. If perplexity blew up, the RTN scaling was too aggressive.

## Bonus: Perplexity sanity checkQuick verification that the quantized model still produces sensible language modeling outputs.

In [ ]:
def compute_perplexity(model, tokens_2d, batch_size=16):    losses = []    for i in range(0, tokens_2d.shape[0], batch_size):        batch = tokens_2d[i:i+batch_size]        loss = model(batch, return_type="loss")        losses.append(loss.item())    avg_loss = np.mean(losses)    return np.exp(avg_loss), avg_loss# FP16 perplexity (weights were restored above)ppl_fp16, loss_fp16 = compute_perplexity(model, tokens_smoke)print(f"FP16: loss={loss_fp16:.4f}, perplexity={ppl_fp16:.2f}")# Re-quantize for the INT8 perplexityfor name, param in model.named_parameters():    if any(s in name for s in ["W_Q", "W_K", "W_V", "W_O", "W_in", "W_out"]):        w = param.data        scale = w.abs().max() / 127.0        q = torch.round(w / scale).clamp(-128, 127)        param.data = (q * scale).to(w.dtype)ppl_int8, loss_int8 = compute_perplexity(model, tokens_smoke)print(f"INT8: loss={loss_int8:.4f}, perplexity={ppl_int8:.2f}")print(f"\nRelative perplexity increase: {(ppl_int8/ppl_fp16 - 1)*100:.2f}%")print("(If < 5%, quantization was mild and the comparison is meaningful.")print(" If > 20%, the quantization is too aggressive — try a less extreme scheme.)")# Restore for any further experimentsmodel.load_state_dict(original_state)